## Convert .wav Files to .pt

In [ ]:
# Imports

import torch
import torchaudio
from pathlib import Path

In [ ]:
# Function to take in .wav files and convert to .pt for encodec 24Hz
# BASE_DIR = Path("/kaggle/input/datasets/matewosendaylalu/encodec-24hz/processed")
# OUT_DIR = Path("/kaggle/working/encodec_24Hz_pt")


# def wav_to_pt(wav_path: str | Path, base_dir: Path, out_dir: Path) -> Path:
#     """Convert a .wav file to a .pt tensor file under out_dir, mirroring the base_dir structure."""
#     src = Path(wav_path)
#     if src.suffix.lower() != ".wav":
#         raise ValueError(f"Expected a .wav file, got: {src.suffix}")

#     waveform, sample_rate = torchaudio.load(src)

#     # Mirror the subdirectory structure from base_dir into out_dir
#     rel = src.relative_to(base_dir)
#     dest = (out_dir / rel).with_suffix(".pt")
#     dest.parent.mkdir(parents=True, exist_ok=True)

#     torch.save({"waveform": waveform, "sample_rate": sample_rate}, dest)
#     # print(f"Saved: {dest}")
#     return dest


# def convert_all(base_dir: str | Path = BASE_DIR, out_dir: str | Path = OUT_DIR) -> list[Path]:
#     """Convert all .wav files in base_dir (recursively) and save .pt files to out_dir."""
#     base_dir, out_dir = Path(base_dir), Path(out_dir)
#     wav_files = list(base_dir.rglob("*.wav"))

#     if not wav_files:
#         print(f"No .wav files found in {base_dir}")
#         return []

#     print(f"Found {len(wav_files)} .wav file(s). Converting...")
#     return [wav_to_pt(f, base_dir, out_dir) for f in wav_files]

# if __name__ == "__main__":
#     convert_all()

In [ ]:
# Function to take in .wav files and convert to .pt for CodecFake Dev
BASE_DIR = Path("/kaggle/input/datasets/jebetk/codecfake-dev/dev_split/dev")
OUT_DIR = Path("/kaggle/working/codecfake_pt")


def wav_to_pt(wav_path: str | Path, base_dir: Path, out_dir: Path) -> Path:
    """Convert a .wav file to a .pt tensor file under out_dir, mirroring the base_dir structure."""
    src = Path(wav_path)
    if src.suffix.lower() != ".wav":
        raise ValueError(f"Expected a .wav file, got: {src.suffix}")

    waveform, sample_rate = torchaudio.load(src)

    # Mirror the subdirectory structure from base_dir into out_dir
    rel = src.relative_to(base_dir)
    dest = (out_dir / rel).with_suffix(".pt")
    dest.parent.mkdir(parents=True, exist_ok=True)

    torch.save({"waveform": waveform, "sample_rate": sample_rate}, dest)
    # print(f"Saved: {dest}")
    return dest


def convert_all(base_dir: str | Path = BASE_DIR, out_dir: str | Path = OUT_DIR) -> list[Path]:
    """Convert all .wav files in base_dir (recursively) and save .pt files to out_dir."""
    base_dir, out_dir = Path(base_dir), Path(out_dir)
    wav_files = list(base_dir.rglob("*.wav"))

    if not wav_files:
        print(f"No .wav files found in {base_dir}")
        return []

    print(f"Found {len(wav_files)} .wav file(s). Converting...")
    return [wav_to_pt(f, base_dir, out_dir) for f in wav_files]

if __name__ == "__main__":
    convert_all()

In [ ]:
# Delete asvspoof dir
# !rm -rf /kaggle/working/for_original_fake_pt*

In [ ]:
# Convert ASVspoof2021 .flac to .pt
from pathlib import Path
import torch
import torchaudio

SUPPORTED_EXTENSIONS = {".wav", ".flac"}

def audio_to_pt(src: str | Path, base_dir: Path, out_dir: Path) -> Path:
    """Convert a .wav or .flac file to a .pt tensor file under out_dir, mirroring the base_dir structure."""
    src = Path(src)
    if src.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported format: {src.suffix}. Supported: {SUPPORTED_EXTENSIONS}")

    waveform, sample_rate = torchaudio.load(src)

    rel = src.relative_to(base_dir)
    dest = (out_dir / rel).with_suffix(".pt")
    dest.parent.mkdir(parents=True, exist_ok=True)

    torch.save({"waveform": waveform, "sample_rate": sample_rate}, dest)
    return dest

def convert_first_n(base_dir: str | Path, out_dir: str | Path, n: int) -> list[Path]:
    """Convert first n audio files in base_dir to .pt."""
    base_dir, out_dir = Path(base_dir), Path(out_dir)
    
    converted = []
    for ext in SUPPORTED_EXTENSIONS:
        for f in base_dir.rglob(f"*{ext}"):
            if len(converted) >= n:
                print(f"Done! Converted {len(converted)} files")
                return converted
            converted.append(audio_to_pt(f, base_dir, out_dir))
            if len(converted) % 1000 == 0:
                print(f"Converted {len(converted)}/{n} files")
    
    print(f"Done! Converted {len(converted)} files")
    return converted

if __name__ == "__main__":
    INPUT_DIR = Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021/ASVspoof2021_LA_eval/ASVspoof2021_LA_eval/flac")
    PT_DIR = Path("/kaggle/working/asvspoof_pt")

    # Convert first 5600 files
    convert_first_n(base_dir=INPUT_DIR, out_dir=PT_DIR, n=5_600)

In [ ]:
# Convert FOR to .pt
from pathlib import Path
import torch
import torchaudio
import random

SUPPORTED_EXTENSIONS = {".wav", ".flac", ".mp3"}
failed_files = []

def audio_to_pt(src: str | Path, base_dir: Path, out_dir: Path) -> Path | None:
    """Convert audio file to .pt tensor file under out_dir, mirroring the base_dir structure."""
    src = Path(src)
    if src.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError(f"Unsupported format: {src.suffix}. Supported: {SUPPORTED_EXTENSIONS}")

    waveform, sample_rate = torchaudio.load(src)

    rel = src.relative_to(base_dir)
    dest = (out_dir / rel).with_suffix(".pt")
    dest.parent.mkdir(parents=True, exist_ok=True)

    torch.save(waveform, dest)
    return dest

def safe_audio_to_pt(src: Path, base_dir: Path, out_dir: Path) -> Path | None:
    """Wrapper that catches errors instead of crashing."""
    try:
        return audio_to_pt(src, base_dir, out_dir)
    except Exception as e:
        print(f"Skipping {src.name}: {e}")
        failed_files.append(src)
        return None

def convert_first_n(base_dir: str | Path, out_dir: str | Path, n: int, shuffle: bool = False) -> list[Path]:
    """Convert first n audio files in base_dir to .pt. Optionally shuffle before taking n."""
    base_dir, out_dir = Path(base_dir), Path(out_dir)
    
    # Collect files incrementally
    all_files = []
    converted = []
    
    print(f"Looking for files in {base_dir}...")
    
    for ext in SUPPORTED_EXTENSIONS:
        for f in base_dir.rglob(f"*{ext}"):
            all_files.append(f)
            
            # If we have enough files and want to shuffle, continue collecting
            if not shuffle and len(all_files) >= n:
                break
        
        if not shuffle and len(all_files) >= n:
            break
    
    print(f"Found {len(all_files)} total files")
    
    # Shuffle if requested
    if shuffle:
        random.seed(42)
        selected = random.sample(all_files, min(n, len(all_files)))
    else:
        selected = all_files[:n]
    
    print(f"Converting {len(selected)} files...")
    
    # Convert selected files
    for i, f in enumerate(selected):
        result = safe_audio_to_pt(f, base_dir, out_dir)
        if result:
            converted.append(result)
        
        if (i + 1) % 1000 == 0:
            print(f"Progress: {i + 1}/{len(selected)} files converted")
    
    # Report failures
    if failed_files:
        print(f"\n{len(failed_files)} file(s) failed and were skipped:")
        for f in failed_files[:10]:  # Show first 10 failures
            print(f"  {f}")
        if len(failed_files) > 10:
            print(f"  ... and {len(failed_files) - 10} more")
    
    print(f"\nDone! Successfully converted {len(converted)} files")
    return converted

# --- Example usage ---
if __name__ == "__main__":
    INPUT_DIR = Path("/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-original/for-original/training/fake")
    PT_DIR = Path("/kaggle/working/for_original_fake_pt")

    # Convert first 5600 files (fastest)
    convert_first_n(base_dir=INPUT_DIR, out_dir=PT_DIR, n=5_600)
    
    # Or convert 5600 random files (slower but still incremental)
    # convert_first_n(base_dir=INPUT_DIR, out_dir=PT_DIR, n=5_600, shuffle=True)

In [ ]:
# Check pt files count
for part in ["codecfake_pt", "for_original_fake_pt", "asvspoof_pt"]:
    pt_files = list(Path(f"/kaggle/working/{part}").rglob("*.pt"))
    print(f"{part}: {len(pt_files)} .pt files")

# Generate Dummy Data

In [ ]:
# Delete everything inside /kaggle/working
# !rm -rf /kaggle/working/*

In [ ]:
# import os
# import torch
# import random
# import pandas as pd
# from pathlib import Path
# from sklearn.model_selection import train_test_split

# SEED = 42
# random.seed(SEED)

# # ── 1. Config (mirrors the real prepare_ADA_splits) ──────────────────────────
# BASE = Path("/kaggle/working")

# dataset_roots = {
#     "CodecFake":    BASE / "audio/CodecFake/fake",
#     "ASVspoof2021": BASE / "audio/ASVspoof2021/fake",
#     "FakeOrReal":   BASE / "audio/FOR/fake"
# }
# dataset_labels = {
#     "CodecFake": 0,
#     "ASVspoof2021": 1,
#     "FakeOrReal": 2
# }

# N_PER_DATASET   = 50       # small enough to be instant
# SAMPLE_RATE     = 16000
# DURATION_S      = 2
# N_SAMPLES       = SAMPLE_RATE * DURATION_S   # [1, 32000] tensors

# # ── 2. Generate .pt files ─────────────────────────────────────────────────────
# for name, root in dataset_roots.items():
#     root.mkdir(parents=True, exist_ok=True)
    
#     if name == "CodecFake":
#         # Distribute evenly across F01-F06
#         for i in range(N_PER_DATASET):
#             cls = (i % 6) + 1          # cycles 1→2→3→4→5→6→1→...
#             tensor = torch.randn(1, N_SAMPLES).float()
#             torch.save(tensor, root / f"F0{cls}_sample_{i:04d}.pt")
#     else:
#         for i in range(N_PER_DATASET):
#             tensor = torch.randn(1, N_SAMPLES).float()
#             torch.save(tensor, root / f"sample_{i:04d}.pt")
    
#     print(f"Generated {N_PER_DATASET} .pt files → {root}")

# AUTOENCODER CODE

## Autoencoder Training

In [ ]:
import re
from torch.utils.data import Dataset

class CodecFakeMultiClassDataset(Dataset):
    """
    PyTorch Dataset for loading CodecFake deepfake audio samples with automatic class extraction.
    
    This dataset is specifically designed for the CodecFake dataset, which contains deepfake
    audio samples generated by different models. It automatically extracts class labels from
    filename patterns and provides samples ready for multi-class classification tasks.
    
    The dataset expects files named with pattern containing "F01", "F02", ..., "F06" where
    the number indicates the generation model used (6 different deepfake models).
    
    Parameters:
    -----------
    root_dir : str or Path
        Root directory containing .pt files (preprocessed audio tensors)
    seed : int, optional (default=None)
        Random seed for reproducible shuffling of samples
        If None, uses system random state
        
    File Naming Convention:
    -----------------------
    Files should follow pattern: "...F{model_number}_..."
    Examples:
    - sample_F01_audio.pt → Class 1 (model 1)
    - sample_F02_audio.pt → Class 2 (model 2)
    - sample_F06_audio.pt → Class 6 (model 6)
    
    Label Processing:
    -----------------
    - Extracts labels 1-6 from filenames
    - Automatically converts to 0-based indexing (1→0, 2→1, ..., 6→5)
    - Suitable for PyTorch classification models expecting 0-based classes
    
    Dataset Statistics:
    -------------------
    Prints during initialization:
    - Number of samples per class (1-6)
    - Total number of samples found
    - Helps verify dataset balance and completeness
    
    Data Format:
    ------------
    - Input files: .pt files containing preprocessed audio tensors
    - Tensors are automatically converted to float type
    - Expected tensor shape: [channels, time_steps] for audio
    
    Returns:
    --------
    tuple: (tensor, label)
        tensor : torch.Tensor
            Audio tensor loaded from .pt file (converted to float)
        label : int
            Zero-based class label (0-5 for models 1-6)
            
    Raises:
    -------
    ValueError
        If no valid .pt files found or unable to extract labels from filenames
        
    Usage:
    ------
    # Basic usage
    dataset = CodecFakeMultiClassDataset(
        root_dir="/path/to/CodecFake_processed/fake_processed"
    )
    
    # With reproducible shuffling
    dataset = CodecFakeMultiClassDataset(
        root_dir="/path/to/CodecFake_processed/fake_processed",
        seed=42
    )
    
    # Use with DataLoader
    from torch.utils.data import DataLoader
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
    
    # Access samples
    audio_tensor, model_label = dataset[0]
    print(f"Audio shape: {audio_tensor.shape}")
    print(f"Generation model: {model_label + 1}")  # Convert back to 1-based
    
    Expected Directory Structure:
    -----------------------------
    root_dir/
    ├── F01_audio.pt  # Model 1 samples
    ├── sample2_F01_audio.pt
    ├── sample3_F02_audio.pt  # Model 2 samples
    ├── sample4_F02_audio.pt
    ├── ...
    └── sampleN_F06_audio.pt  # Model 6 samples
    
    Applications:
    -------------
    - ADMR (Audio Deepfake Model Recognition) tasks
    - Multi-class classification of generation models
    - Training models to identify deepfake generation techniques
    - Forensic analysis of synthetic audio
    """
    def __init__(self, root_dir, seed=None, target_len=80000):
        self.samples = []
        self.target_len = target_len
        root = Path(root_dir)

        for file_path in sorted(root.glob("*.pt")):
            match = re.search(r"F(\d+)_", file_path.name)
            if match:
                label = int(match.group(1))
                self.samples.append((file_path, label))

        # Print the number of samples for each class
        class_counts = {}
        for _, label in self.samples:
            if label in class_counts:
                class_counts[label] += 1
            else:
                class_counts[label] = 1
        for label, count in class_counts.items():
            print(f"Class {label}: {count} samples")
        # Print the total number of samples
        print(f"Total samples: {len(self.samples)}")

        # Shuffle the samples
        if seed is not None:
            rng = random.Random(seed)
            rng.shuffle(self.samples)
        else:
            random.shuffle(self.samples)


        if not self.samples:
            raise ValueError("No valid .pt files found or unable to extract labels.")

    def __len__(self):
        return len(self.samples)

    # def __getitem__(self, idx):
    #     path, label = self.samples[idx]
    #     tensor = torch.load(path).float()
    #     label = label - 1  # Convert to zero-based index for classification tasks
    #     return tensor, label

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        data = torch.load(path)
        # Handle both old dict format and new raw tensor format
        tensor = data["waveform"].float() if isinstance(data, dict) else data.float()
        label = label - 1

        # Pad or truncate to fixed length
        length = tensor.shape[1]
        if length < self.target_len:
            tensor = F.pad(tensor, (0, self.target_len - length))
        else:
            tensor = tensor[:, :self.target_len]
            
        return tensor, label

In [ ]:
import os
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from tqdm import tqdm
from torch.utils.data import DataLoader

In [ ]:
class DeepAutoencoder(nn.Module):
    """
    Deep Convolutional Autoencoder for audio waveform feature learning and reconstruction.
    
    This autoencoder is designed for learning compressed representations of audio waveforms,
    particularly useful for deepfake audio analysis. The model uses 1D convolutional layers
    to process temporal audio data and learns to reconstruct the input waveforms through
    an encoder-decoder architecture.
    
    Architecture:
    -------------
    Encoder: 4 Conv1D layers with progressively increasing channels (1→32→64→128→256)
    - Each layer uses kernel_size=9, stride=2 for downsampling
    - BatchNorm1d + ReLU activation after each convolution
    - Results in compressed latent representation
    
    Decoder: 4 ConvTranspose1D layers with progressively decreasing channels (256→128→64→32→1)
    - Each layer uses kernel_size=9, stride=2 for upsampling
    - BatchNorm1d + ReLU activation (except final layer)
    - Dropout(0.3) before final layer for regularization
    - Final Tanh activation for output normalization
    
    Input/Output:
    -------------
    - Input: [batch_size, 1, sequence_length] - Single-channel audio waveforms
    - Latent: [batch_size, 256, compressed_length] - Learned feature representation
    - Output: [batch_size, 1, sequence_length] - Reconstructed waveforms
    
    Key Features:
    -------------
    - Temporal preservation through 1D convolutions
    - Progressive compression and expansion
    - Batch normalization for stable training
    - Dropout regularization to prevent overfitting
    - Tanh output activation for bounded reconstruction
    
    Applications:
    -------------
    - Feature extraction for downstream tasks (ADA, ADMR)
    - Audio compression and denoising
    - Anomaly detection in audio data
    - Transfer learning for audio classification tasks
    
    Usage:
    ------
    # Create and use autoencoder
    autoencoder = DeepAutoencoder()
    
    # Forward pass
    reconstructed = autoencoder(audio_waveforms)
    
    # Access encoder for feature extraction
    features = autoencoder.encoder(audio_waveforms)
    """
    def __init__(self):
        super(DeepAutoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 256, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )

        # Decoder with dropout
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(256, 128, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.ConvTranspose1d(128, 64, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.ConvTranspose1d(64, 32, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Dropout(0.3),
            nn.ConvTranspose1d(32, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

In [ ]:
# def train(model, train_loader, val_loader, device, epochs=50, lr=1e-4, save_path='/kaggle/working/models/autoencoder.pt'):
#     """
#     Trains the Deep Autoencoder with validation monitoring and early stopping.
    
#     This function implements a complete training loop for autoencoder training using
#     reconstruction loss. The model learns to compress and reconstruct audio waveforms,
#     with automatic saving of the best model based on validation performance.
    
#     Parameters:
#     -----------
#     model : DeepAutoencoder
#         The autoencoder model to train
#     train_loader : DataLoader
#         DataLoader containing training audio samples
#     val_loader : DataLoader
#         DataLoader containing validation audio samples for monitoring
#     device : torch.device
#         Device for training ('cuda' or 'cpu')
#     epochs : int, optional (default=50)
#         Maximum number of training epochs
#     lr : float, optional (default=1e-4)
#         Learning rate for Adam optimizer
#     save_path : str, optional
#         Path where the best model weights will be saved
        
#     Training Configuration:
#     -----------------------
#     - Optimizer: Adam with specified learning rate
#     - Loss Function: SmoothL1Loss(beta=0.0001) for robust reconstruction
#         * More robust to outliers than MSE
#         * Smooth transition between L1 and L2 loss behavior
#     - Early Stopping: Saves model when validation loss improves
#     - Batch Processing: Processes audio samples in batches
    
#     Training Process:
#     -----------------
#     1. Training Phase:
#        - Set model to training mode
#        - Forward pass: reconstruction = model(audio)
#        - Compute reconstruction loss
#        - Backpropagation and parameter updates
       
#     2. Validation Phase:
#        - Set model to evaluation mode (no gradients)
#        - Compute validation reconstruction loss
#        - Save model if validation loss improved
       
#     3. Progress Monitoring:
#        - Print training and validation loss each epoch
#        - Save confirmation when model improves
    
#     Output:
#     -------
#     Saves best model state dict to save_path when validation loss improves
    
#     Loss Function Details:
#     ----------------------
#     SmoothL1Loss provides better training stability compared to MSE:
#     - Behaves like L2 loss for small errors (smooth gradients)
#     - Behaves like L1 loss for large errors (robust to outliers)
#     - Beta=0.0001 controls the transition point
    
#     Usage:
#     ------
#     # Train autoencoder
#     train(
#         model=autoencoder,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         device=torch.device("cuda"),
#         epochs=50,
#         lr=1e-4,
#         save_path="/path/to/models/autoencoder.pt"
#     )
    
#     # Load trained model for inference or transfer learning
#     autoencoder.load_state_dict(torch.load("/path/to/models/autoencoder.pt"))
#     """
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     criterion = nn.SmoothL1Loss(beta=0.0001)
#     best_val_loss = float('inf')

#     for epoch in range(epochs):
#         model.train()
#         train_loss = 0.0
#         for x, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
#             x = x.to(device)
#             optimizer.zero_grad()
#             out = model(x)
#             loss = criterion(out, x)
#             loss.backward()
#             optimizer.step()
#             train_loss += loss.item()

#         model.eval()
#         val_loss = 0.0
#         with torch.no_grad():
#             for x, _ in val_loader:
#                 x = x.to(device)
#                 out = model(x)
#                 loss = criterion(out, x)
#                 val_loss += loss.item()

#         avg_train_loss = train_loss / len(train_loader)
#         avg_val_loss = val_loss / len(val_loader)

#         print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             os.makedirs(os.path.dirname(save_path), exist_ok=True)
#             if os.path.exists(save_path):
#                 os.remove(save_path)
#             torch.save(model.state_dict(), save_path)
#             print(f"Model saved to {save_path}")

In [ ]:
def train_with_checkpoints(model, train_loader, val_loader, lr, epochs, save_path, 
                          resume_path=None, save_freq=5):
    device = next(model.parameters()).device
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.SmoothL1Loss(beta=0.0001)
    
    start_epoch = 0
    best_val_loss = float('inf')
    
    if resume_path and os.path.exists(resume_path):
        checkpoint = torch.load(resume_path)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_loss']
        print(f"Resumed from epoch {start_epoch}")
    
    os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
    
    for epoch in range(start_epoch, epochs):
        # Training
        model.train()
        train_loss = 0
        for x, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), x)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, _ in val_loader:
                x = x.to(device)
                val_loss += criterion(model(x), x).item()
        
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        print(f"Epoch {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")
        
        # Save best model
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), save_path)
        
        # Save checkpoint
        if (epoch + 1) % save_freq == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_loss': best_val_loss,
            }, f'checkpoint_epoch_{epoch+1}.pt')

In [ ]:
# Read codecfake_pt sample


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_dir = "/kaggle/working"

dataset = CodecFakeMultiClassDataset(
    root_dir="/kaggle/working/codecfake_pt"
)

model_dir = Path("/kaggle/working/models")
model_dir.mkdir(parents=True, exist_ok=True)

# Split dataset into training and validation sets
total_len = len(dataset)
train_len = int(0.8 * total_len)
val_len = total_len - train_len
seed = 42  # Set a seed for reproducibility
generator = torch.Generator().manual_seed(seed)
train_set, val_set = data.random_split(dataset, [train_len, val_len], generator=generator)

print(f"Total samples: {total_len}")
print(f"Training samples: {len(train_set)}")
print(f"Validation samples: {len(val_set)}")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)

model = DeepAutoencoder().to(device)
save_path = '/kaggle/working/models/autoencoder.pt'
print(f"Model will be saved to {save_path}")
train(model, train_loader, val_loader, device, epochs=10, lr=1e-4, save_path=save_path)

# Audio Deepfake Attribution (ADA)

## Prepare ADA Data Splits

In [ ]:
import os
import random
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

def prepare_ADA_splits():
    """
    Prepares a dataset split for Audio Deepfake Attribution (ADA) tasks by collecting samples 
    from multiple datasets, ensuring a balanced number of samples per dataset, and saving 
    the splits to CSV files.
    
    This function creates a multi-class dataset for attribution tasks where each class 
    represents a different source dataset. It samples up to MAX_SAMPLES_PER_DATASET 
    from each source and creates stratified train/validation/test splits.
    
    Parameters:
    -----------
    None (uses hardcoded configuration within the function)
    
    Configuration:
    --------------
    - dataset_roots: Dictionary mapping dataset names to their root directories
    - dataset_labels: Dictionary mapping dataset names to integer labels (0, 1, 2)
    - MAX_SAMPLES_PER_DATASET: Maximum number of samples to use from each dataset (25000)
    - output_csv_dir: Directory where CSV splits will be saved
    
    Dataset Structure Expected:
    ---------------------------
    Each dataset root should contain .pt files (PyTorch tensors) organized in subdirectories.
    The function recursively searches for all .pt files in the root directory.
    
    Output:
    -------
    Creates three CSV files in the output directory:
    - train.csv: 60% of the data for training
    - val.csv: 20% of the data for validation  
    - test.csv: 20% of the data for testing
    
    Each CSV file contains two columns:
    - 'path': Absolute path to the .pt file
    - 'label': Integer label indicating the source dataset (0, 1, or 2)
    
    The splits are stratified to ensure balanced representation of all classes.
    
    Usage:
    ------
    prepare_ADA_splits()
    
    Example Output Structure:
    -------------------------
    /path/to/csv/ADA_split/
    ├── train.csv  (60% of samples from all datasets)
    ├── val.csv    (20% of samples from all datasets)
    └── test.csv   (20% of samples from all datasets)
    """
    
    # paths
    dataset_roots = {
        "CodecFake": "/kaggle/working/codecfake_pt",
        "ASVspoof2021": "/kaggle/working/asvspoof_pt",
        "FakeOrReal": "/kaggle/working/for_original_fake_pt"
    }

    # Labels
    dataset_labels = {
        "CodecFake": 0,
        "ASVspoof2021": 1,
        "FakeOrReal": 2
    }

    # Quotas to use for each dataset
    MAX_SAMPLES_PER_DATASET = 25000

    # Output
    output_csv_dir = "/kaggle/working/csv/ADA_split"
    os.makedirs(output_csv_dir, exist_ok=True)

    # Collect samples from all datasets
    all_samples = []

    for name, root in dataset_roots.items():
        files = list(Path(root).rglob("*.pt"))
        files = sorted(files)  # deterministic order

        if len(files) < MAX_SAMPLES_PER_DATASET:
            print(f"Warning: {name} has only {len(files)} files. Using all.")
            selected = files
        else:
            selected = random.sample(files, MAX_SAMPLES_PER_DATASET)

        label = dataset_labels[name]
        all_samples.extend([(str(f.resolve()), label) for f in selected])
        print(f"{name}: selected {len(selected)} samples (label={label})")


    # Shuffle and split
    random.shuffle(all_samples)

    train_val, test = train_test_split(all_samples, test_size=0.2, stratify=[l for _, l in all_samples], random_state=SEED)
    train, val = train_test_split(train_val, test_size=0.25, stratify=[l for _, l in train_val], random_state=SEED)
    # Result: 60% train, 20% val, 20% test

    splits = {"train": train, "val": val, "test": test}

    # Saving CSV
    for name, split in splits.items():
        df = pd.DataFrame(split, columns=["path", "label"])
        csv_path = os.path.join(output_csv_dir, f"{name}.csv")
        df.to_csv(csv_path, index=False)
        print(f"{name} split saved to {csv_path} ({len(split)} samples)")


In [ ]:
print("Preparing attribution splits for ADA...")
prepare_ADA_splits()
print("ADA splits prepared successfully.")


## ADA Module Implementation

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
from torch import nn
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import torch.utils.data as data
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class PathLabelDataset(Dataset):
    """
    Custom PyTorch Dataset for loading preprocessed audio samples from CSV files.
    
    This dataset loads audio tensors that have been preprocessed and saved as PyTorch
    .pt files. It reads file paths and corresponding labels from a CSV file and loads
    the tensors on-demand during training/evaluation.
    
    Parameters:
    -----------
    csv_file : str
        Path to CSV file containing 'path' and 'label' columns
        
    CSV Format Expected:
    --------------------
    path,label
    /path/to/sample1.pt,0
    /path/to/sample2.pt,1
    /path/to/sample3.pt,2
    
    Returns:
    --------
    tuple: (tensor, label)
        - tensor: PyTorch tensor of audio features (converted to float)
        - label: Integer class label
        
    Usage:
    ------
    dataset = PathLabelDataset("train.csv")
    dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
    """
    def __init__(self, csv_file, target_len=80000):
        df = pd.read_csv(csv_file)
        self.target_len = target_len
        self.samples = [(Path(p), l) for p, l in zip(df['path'], df['label'])]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        data = torch.load(Path(path).resolve())
        # Extract waveform from dict
        if isinstance(data, dict):
            # ASVspoof format
            tensor = data['waveform'].float()
        else:
            # FOR format (direct tensor)
            tensor = data.float()

        # Pad or truncate to fixed length
        length = tensor.shape[1]
        if length < self.target_len:
            tensor = F.pad(tensor, (0, self.target_len - length))
        else:
            tensor = tensor[:, :self.target_len]
        
        return tensor, label

In [ ]:
class AudioDeepfakeAttributionModel(nn.Module):
    """
    Neural network model for Audio Deepfake Attribution (ADA) tasks.
    
    This model performs multi-class classification to identify which dataset
    a deepfake audio sample originates from. It uses a pretrained autoencoder
    as a feature extractor with an attention mechanism and classification head.
    
    Architecture:
    -------------
    1. Pretrained Encoder (mostly frozen) - Extracts latent features
    2. Attention Module - Conv1d + Sigmoid for feature weighting  
    3. Classification Head - Fully connected layers for final prediction
    
    Parameters:
    -----------
    pretrained_autoencoder : DeepAutoencoder
        Pretrained autoencoder whose encoder will be used as feature extractor
    num_classes : int, optional (default=3)
        Number of source datasets to classify (typically 3 for ADA)
        
    Model Components:
    -----------------
    - encoder: Pretrained encoder with frozen layers except final layer
    - attention: Conv1d(256->256) + Sigmoid for attention weights
    - classifier: AdaptiveAvgPool1d + Linear layers (256->128->num_classes)
    
    Forward Pass:
    -------------
    Input: [batch_size, channels, time_steps]
    1. Extract features: encoder(x) -> [batch_size, 256, time_steps]  
    2. Compute attention: attention(features) -> [batch_size, 256, time_steps]
    3. Apply attention: features * attention_weights
    4. Classify: classifier(attended_features) -> [batch_size, num_classes]
    
    Usage:
    ------
    # Load pretrained autoencoder
    autoencoder = DeepAutoencoder()
    autoencoder.load_state_dict(torch.load("autoencoder.pt"))
    
    # Create ADA model
    model = AudioDeepfakeAttributionModel(autoencoder, num_classes=3)
    
    # Forward pass
    logits = model(audio_batch)
    """
    def __init__(self, pretrained_autoencoder, num_classes=3):
        super().__init__()
        self.encoder = pretrained_autoencoder.encoder
        for name, param in self.encoder.named_parameters():
            if name not in ["10.weight", "10.bias"]:
                param.requires_grad = False

        self.attention = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=1),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        a = self.attention(z)
        z = z * a
        return self.classifier(z)

In [ ]:
# def train_model(model, train_loader, val_loader, device, epochs=20, lr=1e-4, save_path='/kaggle/working/models/ADA_model.pt'):
#     """
#     Trains the Audio Deepfake Attribution model with validation monitoring and early stopping.
    
#     This function implements a complete training loop with validation-based model saving.
#     Only trainable parameters (unfrozen layers) are optimized, making it suitable for
#     transfer learning scenarios.
    
#     Parameters:
#     -----------
#     model : AudioDeepfakeAttributionModel
#         The ADA model to train
#     train_loader : DataLoader  
#         DataLoader for training data
#     val_loader : DataLoader
#         DataLoader for validation data  
#     device : torch.device
#         Device for training ('cuda' or 'cpu')
#     epochs : int, optional (default=20)
#         Maximum number of training epochs
#     lr : float, optional (default=1e-4)
#         Learning rate for Adam optimizer
#     save_path : str, optional
#         Path to save the best model weights
        
#     Training Configuration:
#     -----------------------
#     - Optimizer: Adam (only on trainable parameters)
#     - Loss Function: CrossEntropyLoss  
#     - Early Stopping: Based on validation loss improvement
#     - Metrics: Classification report on validation set each epoch
    
#     Output:
#     -------
#     Saves model state dict to save_path when validation loss improves
    
#     Printed Information:
#     --------------------
#     - Training loss per epoch
#     - Validation classification report (precision, recall, F1)
#     - Model save confirmations
    
#     Usage:
#     ------
#     train_model(
#         model=ada_model,
#         train_loader=train_loader, 
#         val_loader=val_loader,
#         device=torch.device("cuda"),
#         epochs=50,
#         lr=1e-4,
#         save_path="/path/to/models/ADA_model.pt"
#     )
#     """
#     optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
#     criterion = nn.CrossEntropyLoss()
#     best_val_loss = float('inf')

#     for epoch in range(epochs):
#         model.train()
#         total_loss = 0
#         for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
#             x, y = x.to(device), y.to(device)
#             optimizer.zero_grad()
#             loss = criterion(model(x), y)
#             loss.backward()
#             optimizer.step()
#             total_loss += loss.item()
#         print(f"Train Loss: {total_loss / len(train_loader):.4f}")

#         model.eval()
#         val_loss = 0
#         y_true, y_pred = [], []
#         with torch.no_grad():
#             for x, y in val_loader:
#                 x, y = x.to(device), y.to(device)
#                 output = model(x)
#                 val_loss += criterion(output, y).item()
#                 y_true.extend(y.cpu().numpy())
#                 y_pred.extend(output.argmax(dim=1).cpu().numpy())
#         avg_val_loss = val_loss / len(val_loader)
#         print(classification_report(y_true, y_pred, digits=4))

#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             os.makedirs(os.path.dirname(save_path), exist_ok=True)
#             torch.save(model.state_dict(), save_path)
#             print(f"Saved best model to {save_path}")

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=20, lr=1e-4, 
                save_path='/kaggle/working/models/ADA_model.pt',
                checkpoint_dir='/kaggle/working/checkpoints',
                resume_from=None, save_freq=5):
    """
    Trains the Audio Deepfake Attribution model with validation monitoring and checkpoints.
    """
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # Initialize tracking
    start_epoch = 0
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    
    # Resume from checkpoint if specified
    if resume_from and os.path.exists(resume_from):
        checkpoint = torch.load(resume_from)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint.get('best_loss', float('inf'))
        train_losses = checkpoint.get('train_losses', [])
        val_losses = checkpoint.get('val_losses', [])
        print(f"Resumed from epoch {start_epoch} (best val loss: {best_val_loss:.4f})")
    
    # Create directories
    os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    for epoch in range(start_epoch, epochs):
        # Training
        model.train()
        total_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_train = total_loss / len(train_loader)
        train_losses.append(avg_train)
        print(f"Train Loss: {avg_train:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        y_true, y_pred = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                output = model(x)
                val_loss += criterion(output, y).item()
                y_true.extend(y.cpu().numpy())
                y_pred.extend(output.argmax(dim=1).cpu().numpy())
        
        avg_val = val_loss / len(val_loader)
        val_losses.append(avg_val)
        print(classification_report(y_true, y_pred, digits=4))

        # Save best model
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model to {save_path}")

        # Save checkpoint
        if (epoch + 1) % save_freq == 0 or (epoch + 1) == epochs:
            checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pt')
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_loss': best_val_loss,
                'train_losses': train_losses,
                'val_losses': val_losses,
            }, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

# Usage examples:

# Start fresh
# train_model(model, train_loader, val_loader, device, epochs=50, 
#             save_path='/kaggle/working/models/ADA_model.pt')

# # Resume from checkpoint
# train_model(model, train_loader, val_loader, device, epochs=100,
#             save_path='/kaggle/working/models/ADA_model.pt',
#             resume_from='/kaggle/working/checkpoints/checkpoint_epoch_25.pt')

In [ ]:
def evaluate_and_plot(model, loader, device, report_path, plot_dir):
    """
    Comprehensive evaluation of the ADA model with detailed visualizations and metrics.
    
    This function performs thorough model evaluation including classification metrics,
    confusion matrix, ROC curves, precision-recall curves, and t-SNE visualizations
    of the learned latent representations.
    
    Parameters:
    -----------
    model : AudioDeepfakeAttributionModel
        Trained ADA model to evaluate
    loader : DataLoader
        DataLoader containing evaluation data (typically test set)
    device : torch.device  
        Device for inference ('cuda' or 'cpu')
    report_path : str
        Path to save classification report CSV
    plot_dir : str
        Directory to save all visualization plots
        
    Generated Files:
    ----------------
    Classification Report:
    - {report_path}: Detailed classification metrics in CSV format
    
    Visualization Plots (saved to plot_dir):
    - confusion_matrix.png: Confusion matrix heatmap
    - roc_curves.png: ROC curves for all classes
    - pr_curves.png: Precision-Recall curves for all classes  
    - tsne_2d.png: 2D t-SNE of latent representations
    - tsne_3d.png: 3D t-SNE of latent representations
    
    Metrics Computed:
    -----------------
    - Per-class and overall accuracy
    - Precision, recall, F1-score for each class
    - Area Under ROC Curve (AUC) for each class
    - Average Precision (AP) for each class
    - Confusion matrix
    
    Visualizations:
    ---------------
    - ROC Curves: True Positive Rate vs False Positive Rate
    - PR Curves: Precision vs Recall trade-offs
    - t-SNE: Dimensionality reduction of latent features for visualization
    - Confusion Matrix: Classification accuracy breakdown
    
    Usage:
    ------
    evaluate_and_plot(
        model=trained_model,
        loader=test_loader,
        device=torch.device("cuda"), 
        report_path="/path/to/reports/ADA_test_report.csv",
        plot_dir="/path/to/plots/ADA/"
    )
    """
    model.eval()
    y_true, y_pred = [], []
    logits_list = []
    latents = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating"):
            x, y = x.to(device), y.to(device)
            z = model.encoder(x)
            pooled = torch.nn.functional.adaptive_avg_pool1d(z, 1).squeeze(-1)
            latents.append(pooled.cpu())
            out = model(x)
            y_pred.extend(out.argmax(dim=1).cpu().numpy())
            y_true.extend(y.cpu().numpy())
            logits_list.append(out.cpu())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    probs = torch.softmax(torch.cat(logits_list), dim=1).numpy()
    latents = torch.cat(latents, dim=0).numpy()

    report = classification_report(y_true, y_pred, output_dict=True, digits=4)
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    pd.DataFrame(report).transpose().to_csv(report_path)

    matrix = confusion_matrix(y_true, y_pred)
    os.makedirs(plot_dir, exist_ok=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.savefig(os.path.join(plot_dir, "confusion_matrix.png"))
    plt.close()

    # ROC Curves
    y_bin = label_binarize(y_true, classes=list(range(3)))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(3):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(8, 6))
    for i in range(3):
        plt.plot(fpr[i], tpr[i], label=f"Class {i} AUC={roc_auc[i]:.2f}")
    plt.plot([0,1], [0,1], 'k--')
    plt.title("ROC Curves")
    plt.legend()
    plt.savefig(os.path.join(plot_dir, "roc_curves.png"))
    plt.close()

    # Precision-Recall Curves
    precision, recall, ap = {}, {}, {}
    for i in range(3):
        precision[i], recall[i], _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        ap[i] = average_precision_score(y_bin[:, i], probs[:, i])

    plt.figure(figsize=(8, 6))
    for i in range(3):
        plt.plot(recall[i], precision[i], label=f"Class {i} AP={ap[i]:.2f}")
    plt.title("Precision-Recall Curves")
    plt.legend()
    plt.savefig(os.path.join(plot_dir, "pr_curves.png"))
    plt.close()

    # t-SNE 2D
    tsne_2d = TSNE(n_components=2, random_state=SEED, perplexity=1)
    emb_2d = tsne_2d.fit_transform(latents)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=emb_2d[:,0], y=emb_2d[:,1], hue=y_true, palette="tab10", legend="full")
    plt.title("t-SNE (2D) of Latent Space")
    plt.savefig(os.path.join(plot_dir, "tsne_2d.png"))
    plt.close()

    # t-SNE 3D
    tsne_3d = TSNE(n_components=3, random_state=SEED, perplexity=1)
    emb_3d = tsne_3d.fit_transform(latents)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(emb_3d[:, 0], emb_3d[:, 1], emb_3d[:, 2], c=y_true, cmap="tab10")
    legend = ax.legend(*scatter.legend_elements(), title="Class")
    ax.add_artist(legend)
    ax.set_title("t-SNE (3D) of Latent Space")
    plt.savefig(os.path.join(plot_dir, "tsne_3d.png"))
    plt.close()

In [ ]:
def compute_confidence_scores_with_preds(model, loader, device, output_csv="/kaggle/working/csv/confidence_scores.csv"):
    """
    Computes confidence scores and predictions for all samples in the dataset.
    
    This function evaluates the model on provided data and extracts confidence scores
    (maximum softmax probability) along with predicted and true labels. The results
    are saved to a CSV file for further analysis and threshold optimization.
    
    Parameters:
    -----------
    model : AudioDeepfakeAttributionModel
        Trained model to evaluate
    loader : DataLoader
        DataLoader containing samples to analyze
    device : torch.device
        Device for inference ('cuda' or 'cpu')
    output_csv : str, optional
        Path to save the confidence scores CSV file
        
    Process:
    --------
    1. Set model to evaluation mode
    2. For each batch, compute softmax probabilities
    3. Extract maximum confidence and corresponding prediction
    4. Collect true labels, predictions, and confidence scores
    5. Save results to CSV file
    
    Output CSV Format:
    ------------------
    confidence,true_label,pred_label
    0.9123,0,0
    0.7834,1,1  
    0.5621,2,1
    
    Usage:
    ------
    compute_confidence_scores_with_preds(
        model=trained_model,
        loader=train_loader,
        device=torch.device("cuda"),
        output_csv="/path/to/csv/ADA/confidence_scores.csv"
    )
    
    Applications:
    -------------
    - Threshold optimization for reliable predictions
    - Model calibration analysis
    - Uncertainty quantification
    - Performance analysis across confidence levels
    """
    model.eval()
    scores, true_labels, pred_labels = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Computing confidence scores"):
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            max_confidence, pred = probs.max(dim=1)
            
            scores.extend(max_confidence.cpu().numpy())
            pred_labels.extend(pred.cpu().numpy())
            true_labels.extend(y.numpy())

    df = pd.DataFrame({
        "confidence": scores,
        "true_label": true_labels,
        "pred_label": pred_labels
    })
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Confidence scores with predictions saved to {output_csv}")

In [ ]:
def find_confidence_thresholds(csv_path):
    """
    Analyzes confidence scores to find optimal threshold for reliable predictions.
    
    This function reads confidence scores from a CSV file and analyzes the
    accuracy-coverage trade-off to find the best confidence threshold that
    maintains high accuracy while providing reasonable coverage.
    
    Parameters:
    -----------
    csv_path : str
        Path to CSV file containing confidence scores, true labels, and predictions
        (generated by compute_confidence_scores_with_preds)
        
    Analysis Process:
    -----------------
    1. Load confidence scores and labels from CSV
    2. Test thresholds from 0.0 to 1.0 (500 points)
    3. For each threshold, compute:
       - Accuracy on samples above threshold
       - Coverage (percentage of samples above threshold)
    4. Find threshold with ≥80% coverage and maximum accuracy
    
    Output:
    -------
    Returns the optimal threshold value for ≥80% coverage
    
    Printed Results:
    ----------------
    - Coverage ≥80% Threshold and corresponding accuracy
    
    Metrics Explanation:
    --------------------
    - Coverage: Percentage of samples that exceed the confidence threshold
    - Accuracy: Classification accuracy on samples exceeding threshold
    - Trade-off: Higher thresholds = higher accuracy but lower coverage
    
    Usage:
    ------
    threshold = find_confidence_thresholds("/path/to/csv/confidence_scores.csv")
    
    Use Case:
    ---------
    In production, you can reject predictions below this threshold as "uncertain"
    to maintain high accuracy on accepted predictions.
    """
    import matplotlib.pyplot as plt

    df = pd.read_csv(csv_path)
    confidences = df["confidence"].values
    true_labels = df["true_label"].values
    pred_labels = df["pred_label"].values

    thresholds = np.linspace(0.0, 1.0, 500)
    best_cov = 0
    cov_thresh = 0

    cov_list = []

    for t in thresholds:
        mask = confidences >= t
        if np.sum(mask) == 0:
            continue

        preds = pred_labels[mask]
        targets = true_labels[mask]

        accuracy = np.mean(preds == targets)
        coverage = np.sum(mask) / len(true_labels)

        cov_list.append(coverage)

        if coverage >= 0.80 and accuracy > best_cov:
            best_cov = accuracy
            cov_thresh = t

    print(f"Coverage ≥80% Threshold: {cov_thresh:.4f} | Accuracy: {best_cov:.4f}")
    return cov_thresh

In [ ]:
def predict_with_confidence_threshold(model, inputs, device, threshold):
    """
    Makes predictions with confidence-based filtering for reliable classification.
    
    This function performs inference and returns predictions only when the model's
    confidence exceeds a specified threshold. Low-confidence predictions are marked
    as uncertain (-1), allowing for more reliable deployment in production scenarios.
    
    Parameters:
    -----------
    model : AudioDeepfakeAttributionModel
        Trained model for making predictions
    inputs : torch.Tensor
        Input tensor(s) to classify [batch_size, channels, time_steps]
    device : torch.device
        Device for inference ('cuda' or 'cpu')
    threshold : float
        Minimum confidence threshold (0.0 to 1.0)
        Predictions below this threshold return -1 (uncertain)
        
    Returns:
    --------
    tuple: (predictions, confidences)
        predictions : list
            Predicted class labels (0, 1, 2) or -1 for uncertain predictions
        confidences : numpy.ndarray
            Confidence scores (maximum softmax probabilities) for all samples
            
    Process:
    --------
    1. Set model to evaluation mode
    2. Compute softmax probabilities for inputs
    3. Extract maximum confidence and predicted class
    4. Return prediction if confidence ≥ threshold, else -1
    
    Usage:
    ------
    # Using threshold found by find_confidence_thresholds()
    predictions, confidences = predict_with_confidence_threshold(
        model=trained_model,
        inputs=audio_batch,
        device=torch.device("cuda"),
        threshold=0.85
    )
    
    # Handle results
    for pred, conf in zip(predictions, confidences):
        if pred == -1:
            print(f"Uncertain prediction (confidence: {conf:.3f})")
        else:
            print(f"Predicted class: {pred} (confidence: {conf:.3f})")
            
    Applications:
    -------------
    - Production deployment with uncertainty handling
    - Quality control in automated systems
    - Human-in-the-loop workflows for uncertain cases
    - Maintaining high precision in critical applications
    """
    model.eval()
    inputs = inputs.to(device)
    
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1)
        max_confidence, predicted_class = torch.max(probs, dim=1)

        predictions = []
        confidences = max_confidence.cpu().numpy()

        for conf, pred in zip(confidences, predicted_class.cpu().numpy()):
            if conf >= threshold:
                predictions.append(pred)
            else:
                predictions.append(-1)  # -1 indicates uncertainty

    return predictions, confidences

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_csv = "/kaggle/working/csv/ADA_split"
model_path = "/kaggle/working/models/autoencoder.pt"
model_save_path = "/kaggle/working/models/ADA_model.pt"

autoencoder = DeepAutoencoder().to(device)
autoencoder.load_state_dict(torch.load(model_path, map_location=device))
model = AudioDeepfakeAttributionModel(autoencoder).to(device)

train_set = PathLabelDataset(os.path.join(base_csv, "train.csv"))
val_set = PathLabelDataset(os.path.join(base_csv, "val.csv"))
test_set = PathLabelDataset(os.path.join(base_csv, "test.csv"))

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)
test_loader = DataLoader(test_set, batch_size=16)

train_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4, save_path=model_save_path)
model.load_state_dict(torch.load(model_save_path, map_location=device))

compute_confidence_scores_with_preds(model, train_loader, device,
                                     output_csv="/kaggle/working/csv/ADA/confidence_scores.csv")

find_confidence_thresholds("/kaggle/working/csv/ADA/confidence_scores.csv")

In [ ]:
evaluate_and_plot(
        model=model,
        loader=test_loader,
        device=torch.device("cuda"), 
        report_path="/kaggle/working/reports/ADA_test_report.csv",
        plot_dir="/kaggle/working/plots/ADA/"
    )

# Audio Model Deepfake Recognition

## Prepare ADMR Data Splits

In [ ]:
def prepare_ADMR_splits():
    """
    Prepares a dataset split for Audio Deepfake Model Recognition (ADMR) tasks by collecting 
    samples from the CodecFake dataset and saving the splits to CSV files.
    
    This function creates a multi-class dataset for model recognition tasks where each class 
    represents a different deepfake generation model. It uses the CodecFakeMultiClassDataset 
    to automatically handle the class assignment based on the model used to generate each sample.
    
    Parameters:
    -----------
    None (uses hardcoded configuration within the function)
    
    Configuration:
    --------------
    - output_csv_dir: Directory where CSV splits will be saved ("/path/to/csv/ADMR_split")
    - root_dir: Path to the CodecFake processed fake audio files
    - SEED: Random seed for reproducible splits (42)
    
    Output:
    -------
    Creates three CSV files in the output directory:
    - train.csv: 60% of the data for training
    - val.csv: 20% of the data for validation
    - test.csv: 20% of the data for testing
    
    Each CSV file contains two columns:
    - 'path': Absolute path to the .pt file
    - 'label': Integer label indicating the generation model (1-6, converted to 0-5)
    
    The splits are stratified to ensure balanced representation of all model classes.
    
    Usage:
    ------
    prepare_ADMR_splits()
    
    Example Output Structure:
    -------------------------
    /path/to/csv/ADMR_split/
    ├── train.csv  (60% of samples from all models)
    ├── val.csv    (20% of samples from all models)
    └── test.csv   (20% of samples from all models)
    
    Notes:
    ------
    - Labels are automatically assigned by CodecFakeMultiClassDataset based on directory structure
    - The function assumes 6 different generation models in the CodecFake dataset
    - Uses stratified splitting to maintain class balance across all splits
    - Files should follow naming pattern with F01, F02, ..., F06 for model identification
    """
    # Output
    output_csv_dir = "/kaggle/working/csv/ADMR_split"
    os.makedirs(output_csv_dir, exist_ok=True)

    dataset = CodecFakeMultiClassDataset(
        root_dir="/kaggle/working/codecfake_pt",
        seed=SEED
    )

    # Saving the dataset samples
    all_samples = dataset.samples

    # Splitting the dataset into train, val, and test sets
    train_val, test = train_test_split(all_samples, test_size=0.2, stratify=[lbl for _, lbl in all_samples], random_state=SEED)
    train, val = train_test_split(train_val, test_size=0.25, stratify=[lbl for _, lbl in train_val], random_state=SEED)
    # Result: 60% train, 20% val, 20% test

    splits = {
        "train": train,
        "val": val,
        "test": test
    }

    # Save splits to CSV files
    for name, split in splits.items():
        df = pd.DataFrame([(str(p), l - 1) for p, l in split], columns=["path", "label"])
        save_path = os.path.join(output_csv_dir, f"{name}.csv")
        df.to_csv(save_path, index=False)
        print(f"Saved {name} split with {len(split)} samples.")


In [ ]:
print("\nPreparing attribution splits for ADMR...")
prepare_ADMR_splits()
print("ADMR splits prepared successfully.")

## ADMR Module Implementation

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
from torch import nn
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import torch.utils.data as data
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

class ADMR_model(nn.Module):
    """
    Neural network model for Audio Deepfake Model Recognition (ADMR) tasks.
    
    This model performs multi-class classification to identify which deepfake generation
    model was used to create an audio sample. It uses a pretrained autoencoder as a
    feature extractor with an attention mechanism and classification head for 6-class
    classification (different generation models).
    
    Architecture:
    -------------
    1. Pretrained Encoder (mostly frozen) - Extracts latent audio features
    2. Attention Module - Conv1d + Sigmoid for attention-based feature weighting
    3. Classification Head - Fully connected layers for model recognition
    
    Parameters:
    -----------
    pretrained_autoencoder : DeepAutoencoder
        Pretrained autoencoder whose encoder will be used as feature extractor
        
    Model Components:
    -----------------
    - encoder: Pretrained encoder with frozen layers except final layer ("10.weight", "10.bias")
    - attention: Conv1d(256->256) + Sigmoid for computing attention weights
    - classifier: Sequential layers for 6-class classification
        * AdaptiveAvgPool1d(1): Reduces temporal dimension
        * Flatten(): Flattens pooled features
        * Linear(256, 128): First FC layer with ReLU activation
        * Linear(128, 6): Output layer for 6 generation models
    
    Forward Pass:
    -------------
    Input: [batch_size, channels, time_steps]
    1. Extract features: encoder(x) -> [batch_size, 256, time_steps]
    2. Compute attention: attention(features) -> [batch_size, 256, time_steps]  
    3. Apply attention: features * attention_weights (element-wise)
    4. Classify: classifier(attended_features) -> [batch_size, 6]
    
    Usage:
    ------
    # Load pretrained autoencoder
    autoencoder = DeepAutoencoder()
    autoencoder.load_state_dict(torch.load("autoencoder.pt"))
    
    # Create ADMR model for 6 generation models
    model = ADMR_model(pretrained_autoencoder=autoencoder)
    
    # Forward pass
    logits = model(audio_batch)  # Returns logits for 6 classes
    """
    def __init__(self, pretrained_autoencoder):
        super(ADMR_model, self).__init__()
        self.encoder = pretrained_autoencoder.encoder
        for name, param in self.encoder.named_parameters():
            if name not in ["10.weight", "10.bias"]:
                param.requires_grad = False


        self.attention = nn.Sequential(
            nn.Conv1d(256, 256, kernel_size=1),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 6)  # 6 classes
        )

    def forward(self, x):
        z = self.encoder(x)  # [B, 256, T]
        a = self.attention(z)  # [B, 256, T]
        z = z * a  # Apply attention
        out = self.classifier(z)
        return out
    
    def load_ADMR_model(autoencoder_path, ADMR_model_path, device):
        """
        Static method to load a pretrained ADMR model from disk.
        
        This utility function loads both the pretrained autoencoder and the trained
        ADMR model weights, creating a complete model ready for inference.
        
        Parameters:
        -----------
        autoencoder_path : str
            Path to the pretrained autoencoder .pt file
        ADMR_model_path : str
            Path to the trained ADMR model .pt file
        device : torch.device
            Device to load the model on ('cuda' or 'cpu')
            
        Returns:
        --------
        ADMR_model
            Complete ADMR model loaded and ready for inference (in eval mode)
            
        Usage:
        ------
        device = torch.device("cuda")
        model = ADMR_model.load_ADMR_model(
            autoencoder_path="/path/to/autoencoder.pt",
            ADMR_model_path="/path/to/ADMR_model.pt", 
            device=device
        )
        
        # Now ready for inference
        predictions = model(audio_batch)
        """
        # Loads the pretrained autoencoder
        autoencoder = DeepAutoencoder().to(device)
        autoencoder.load_state_dict(torch.load(autoencoder_path, map_location=device))

        # Loads the pretrained ADMR model
        model = ADMR_model(pretrained_autoencoder=autoencoder).to(device)
        model.load_state_dict(torch.load(ADMR_model_path, map_location=device))
        model.eval()

        print(f"ADMR model loaded from {ADMR_model_path}")
        return model


In [ ]:
def train_ADMR_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4, save_path='/kaggle/working/models/ADMR_model.pt'):
    """
    Trains the ADMR model for deepfake generation model recognition with validation monitoring.
    
    This function implements a complete training loop with validation-based early stopping
    and automatic model saving. Only trainable parameters (unfrozen layers) are optimized,
    making it suitable for transfer learning from pretrained autoencoders.
    
    Parameters:
    -----------
    model : ADMR_model
        The ADMR model to train
    train_loader : DataLoader
        DataLoader containing training data (features, labels)
    val_loader : DataLoader  
        DataLoader containing validation data for performance monitoring
    device : torch.device
        Device for training ('cuda' or 'cpu')
    epochs : int, optional (default=10)
        Maximum number of training epochs
    lr : float, optional (default=1e-4)
        Learning rate for Adam optimizer
    save_path : str, optional
        Path where the best model weights will be saved
        
    Training Configuration:
    -----------------------
    - Optimizer: Adam (only on trainable parameters)
    - Loss Function: CrossEntropyLoss for 6-class classification
    - Early Stopping: Saves model when validation loss improves
    - Metrics: Detailed classification report per epoch
    
    Training Process:
    -----------------
    1. Train for one epoch, compute average training loss
    2. Evaluate on validation set, compute validation loss and metrics
    3. Save model if validation loss improved
    4. Print progress and classification report
    5. Repeat until epochs completed
    
    Output:
    -------
    Saves best model state dict to save_path when validation loss improves
    
    Printed Information:
    --------------------
    - Training and validation loss per epoch
    - Detailed classification report on validation set
    - Model save confirmations with file path
    
    Usage:
    ------
    train_ADMR_model(
        model=admr_model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=torch.device("cuda"),
        epochs=50,
        lr=1e-4,
        save_path="/path/to/models/ADMR_model.pt"
    )
    """
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        y_true, y_pred = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item()
                y_true.extend(y.cpu().numpy())
                y_pred.extend(logits.argmax(dim=1).cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            if os.path.exists(save_path):
                os.remove(save_path)
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

        print(f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(classification_report(y_true, y_pred, digits=4))

In [ ]:
def evaluate_model(model, test_loader, device, report_path="/kaggle/working/reports/ADMR/test_report.csv", plot_path="/kaggle/working/plots/ADMR/"):
    """
    Comprehensive evaluation of the ADMR model with detailed visualizations and metrics.
    
    This function performs thorough model evaluation including classification metrics,
    confusion matrix, ROC curves, precision-recall curves, and t-SNE visualizations
    of the learned latent representations for all 6 generation models.
    
    Parameters:
    -----------
    model : ADMR_model
        Trained ADMR model to evaluate
    test_loader : DataLoader
        DataLoader containing test data for evaluation
    device : torch.device
        Device for inference ('cuda' or 'cpu')
    report_path : str, optional
        Path to save detailed classification report CSV
    plot_path : str, optional  
        Directory path to save all visualization plots
        
    Generated Files:
    ----------------
    Classification Report:
    - {report_path}: Detailed per-class metrics in CSV format
    
    Visualization Plots (saved to plot_path):
    - confusion_matrix.png: 6x6 confusion matrix heatmap with class labels C1-C6
    - tsne_2d.png: 2D t-SNE visualization of latent feature space
    - tsne_3d.png: 3D t-SNE visualization of latent feature space  
    - roc_curves.png: ROC curves for all 6 classes with AUC scores
    - pr_curves.png: Precision-Recall curves for all 6 classes with AP scores
    
    Metrics Computed:
    -----------------
    - Overall test accuracy
    - Per-class precision, recall, F1-score, support
    - Confusion matrix (6x6 for generation models)
    - ROC curves and AUC for each class
    - Precision-Recall curves and Average Precision (AP) for each class
    - t-SNE embeddings for latent space visualization
    
    Evaluation Process:
    -------------------
    1. Extract latent features from encoder for t-SNE visualization
    2. Compute predictions and collect true labels
    3. Calculate all classification metrics
    4. Generate and save all visualization plots
    5. Save detailed classification report
    
    Usage:
    ------
    evaluate_model(
        model=trained_admr_model,
        test_loader=test_loader,
        device=torch.device("cuda"),
        report_path="/path/to/reports/ADMR/test_report.csv",
        plot_path="/path/to/plots/ADMR/"
    )
    
    Output Interpretation:
    ----------------------
    - Classes C1-C6 represent different deepfake generation models
    - Higher AUC values indicate better discrimination for that model
    - t-SNE plots show how well the model separates different generation models
    - Confusion matrix reveals which models are most often confused
    """
    model.eval()
    y_true, y_pred = [], []
    logits_list = []
    latent_features = []

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Evaluating on test set"):
            x, y = x.to(device), y.to(device)
            z = model.encoder(x)  # shape: [B, 256, T]
            pooled = torch.nn.functional.adaptive_avg_pool1d(z, 1).squeeze(-1)  # shape: [B, 256]
            latent_features.append(pooled.cpu())

            logits = model(x)
            preds = logits.argmax(dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            logits_list.append(logits.cpu())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    logits_all = torch.cat(logits_list, dim=0)
    probs = torch.softmax(logits_all, dim=1).numpy()
    latents = torch.cat(latent_features, dim=0).numpy()

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, digits=4)
    matrix = confusion_matrix(y_true, y_pred)

    # Print and save classification report
    print(f"\nTest Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    pd.DataFrame(report).transpose().to_csv(report_path)
    print(f"Classification report saved to {report_path}")

    # Confusion matrix
    cm_path = os.path.join(plot_path, "confusion_matrix.png")
    os.makedirs(os.path.dirname(cm_path), exist_ok=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=[f"C{i+1}" for i in range(6)], yticklabels=[f"C{i+1}" for i in range(6)])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.savefig(cm_path)
    plt.close()
    print(f"Confusion matrix saved to {cm_path}")

    # --- t-SNE 2D ---
    tsne_2d_path = os.path.join(plot_path, "tsne_2d.png")
    tsne_2d = TSNE(n_components=2, random_state=SEED, perplexity=1)
    emb_2d = tsne_2d.fit_transform(latents)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=emb_2d[:, 0], y=emb_2d[:, 1], hue=y_true, palette="tab10", legend="full")
    plt.title("t-SNE (2D) of Test Latent Representations")
    plt.savefig(tsne_2d_path)
    plt.close()
    print(f"t-SNE 2D plot saved to {tsne_2d_path}")

    # --- t-SNE 3D ---
    tsne_3d_path = os.path.join(plot_path, "tsne_3d.png")
    tsne_3d = TSNE(n_components=3, random_state=SEED, perplexity=1)
    emb_3d = tsne_3d.fit_transform(latents)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(emb_3d[:, 0], emb_3d[:, 1], emb_3d[:, 2], c=y_true, cmap="tab10")
    legend = ax.legend(*scatter.legend_elements(), title="Class")
    ax.add_artist(legend)
    ax.set_title("t-SNE (3D) of Test Latent Representations")
    plt.savefig(tsne_3d_path)
    plt.close()
    print(f"t-SNE 3D plot saved to {tsne_3d_path}")

    # --- ROC & PR Curves ---
    y_bin = label_binarize(y_true, classes=list(range(6)))
    fpr, tpr, roc_auc = dict(), dict(), dict()
    precision, recall, ap = dict(), dict(), dict()

    for i in range(6):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        precision[i], recall[i], _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        ap[i] = average_precision_score(y_bin[:, i], probs[:, i])

    # ROC Curve
    roc_path = "/kaggle/working/plots/ADMR/roc_curves.png"
    plt.figure(figsize=(8, 6))
    for i in range(6):
        plt.plot(fpr[i], tpr[i], label=f"C{i+1} (AUC={roc_auc[i]:.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves")
    plt.legend()
    plt.savefig(roc_path)
    plt.close()
    print(f"ROC curves saved to {roc_path}")

    # PR Curve
    pr_path = "/kaggle/working/plots/ADMR/pr_curves.png"
    plt.figure(figsize=(8, 6))
    for i in range(6):
        plt.plot(recall[i], precision[i], label=f"C{i+1} (AP={ap[i]:.2f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curves")
    plt.legend()
    plt.savefig(pr_path)
    plt.close()
    print(f"Precision-Recall curves saved to {pr_path}")

In [ ]:
def compute_confidence_scores_with_preds(model, loader, device, output_csv="/kaggle/working/csv/ADMR/confidence_scores.csv"):
    """
    Computes confidence scores and predictions for ADMR model evaluation and calibration.
    
    This function evaluates the ADMR model on provided data and extracts confidence scores
    (maximum softmax probability) along with predicted and true labels. Results are saved
    for threshold optimization and model calibration analysis.
    
    Parameters:
    -----------
    model : ADMR_model
        Trained ADMR model to evaluate
    loader : DataLoader
        DataLoader containing samples to analyze (typically training set)
    device : torch.device
        Device for inference ('cuda' or 'cpu')
    output_csv : str, optional
        Path to save the confidence scores CSV file
        
    Process:
    --------
    1. Set model to evaluation mode (disable dropout/batchnorm training)
    2. For each batch, compute softmax probabilities over 6 classes
    3. Extract maximum confidence score and corresponding prediction
    4. Collect true labels (0-5), predictions (0-5), and confidence scores
    5. Save all results to structured CSV file
    
    Output CSV Format:
    ------------------
    confidence,true_label,pred_label
    0.9456,0,0
    0.8234,1,1
    0.6789,2,1
    0.9876,5,5
    
    CSV Columns:
    ------------
    - confidence: Maximum softmax probability (0.0 to 1.0)
    - true_label: Ground truth class (0-5 for 6 generation models)
    - pred_label: Model prediction (0-5 for 6 generation models)
    
    Usage:
    ------
    compute_confidence_scores_with_preds(
        model=trained_admr_model,
        loader=train_loader,
        device=torch.device("cuda"),
        output_csv="/path/to/csv/ADMR/confidence_scores.csv"
    )
    """
    model.eval()
    scores, true_labels, pred_labels = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Computing confidence scores"):
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            max_confidence, pred = probs.max(dim=1)
            
            scores.extend(max_confidence.cpu().numpy())
            pred_labels.extend(pred.cpu().numpy())
            true_labels.extend(y.numpy())

    df = pd.DataFrame({
        "confidence": scores,
        "true_label": true_labels,
        "pred_label": pred_labels
    })
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Confidence scores with predictions saved to {output_csv}")

In [ ]:
def find_confidence_thresholds(csv_path):
    """
    Analyzes confidence scores to find optimal threshold for reliable ADMR predictions.
    
    This function reads confidence scores from CSV and analyzes the accuracy-coverage
    trade-off to find the best confidence threshold that maintains high accuracy while
    providing reasonable coverage for generation model recognition tasks.
    
    Parameters:
    -----------
    csv_path : str
        Path to CSV file containing confidence scores, true labels, and predictions
        (generated by compute_confidence_scores_with_preds)
        
    Analysis Process:
    -----------------
    1. Load confidence scores and labels from CSV file
    2. Test 500 threshold values from 0.0 to 1.0
    3. For each threshold, compute:
       - Accuracy on samples exceeding the threshold
       - Coverage (percentage of samples exceeding threshold)
    4. Find optimal threshold with ≥80% coverage and maximum accuracy
    
    Metrics Explanation:
    --------------------
    - Coverage: Percentage of test samples that exceed confidence threshold
    - Accuracy: Classification accuracy on samples exceeding threshold
    - Trade-off: Higher thresholds → higher accuracy but lower coverage
    
    Output:
    -------
    Returns the optimal confidence threshold for ≥80% coverage
    
    Printed Results:
    ----------------
    - Coverage ≥80% Threshold value and corresponding accuracy
    
    Usage:
    ------
    optimal_threshold = find_confidence_thresholds(
        "/path/to/csv/ADMR/confidence_scores.csv"
    )
    
    Production Use:
    ---------------
    In production systems, predictions below this threshold can be:
    - Flagged for human review
    - Rejected as uncertain
    - Processed with alternative methods
    - Used to trigger additional verification steps
    
    ADMR-Specific Considerations:
    -----------------------------
    - Different generation models may have different confidence patterns
    - Some models might be inherently harder to distinguish
    - Threshold should balance reliability with practical coverage needs
    """
    import matplotlib.pyplot as plt

    df = pd.read_csv(csv_path)
    confidences = df["confidence"].values
    true_labels = df["true_label"].values
    pred_labels = df["pred_label"].values

    thresholds = np.linspace(0.0, 1.0, 500)
    best_cov = 0
    cov_thresh = 0

    cov_list = []

    for t in thresholds:
        mask = confidences >= t
        if np.sum(mask) == 0:
            continue

        preds = pred_labels[mask]
        targets = true_labels[mask]

        accuracy = np.mean(preds == targets)
        coverage = np.sum(mask) / len(true_labels)

        cov_list.append(coverage)

        if coverage >= 0.80 and accuracy > best_cov:
            best_cov = accuracy
            cov_thresh = t

    print(f"Coverage ≥80% Threshold: {cov_thresh:.4f} | Accuracy: {best_cov:.4f}")
    return cov_thresh

In [ ]:
def predict_with_confidence_threshold(model, inputs, device, threshold=0.95): #0.8
    """
    Makes ADMR predictions with confidence-based filtering for reliable generation model identification.
    
    This function performs inference on audio samples and returns predictions only when
    the model's confidence exceeds a specified threshold. Low-confidence predictions are
    marked as uncertain (-1), enabling reliable deployment in production scenarios where
    accurate generation model identification is critical.
    
    Parameters:
    -----------
    model : ADMR_model
        Trained ADMR model for generation model recognition
    inputs : torch.Tensor
        Input audio tensor(s) to classify [batch_size, channels, time_steps]
    device : torch.device
        Device for inference ('cuda' or 'cpu')
    threshold : float, optional (default=0.95)
        Minimum confidence threshold (0.0 to 1.0)
        Predictions below this threshold return -1 (uncertain)
        Default 0.95 is quite conservative for high-precision applications
        
    Returns:
    --------
    tuple: (predictions, confidences)
        predictions : list
            Predicted generation model labels:
            - 0-5: Confident predictions for models 1-6
            - -1: Uncertain predictions below threshold
        confidences : numpy.ndarray
            Confidence scores (maximum softmax probabilities) for all samples
            
    Process:
    --------
    1. Set model to evaluation mode
    2. Compute softmax probabilities over 6 generation models
    3. Extract maximum confidence and predicted model class
    4. Return prediction if confidence ≥ threshold, else -1 (uncertain)
    
    Usage:
    ------
    # Using optimized threshold from find_confidence_thresholds()
    predictions, confidences = predict_with_confidence_threshold(
        model=trained_admr_model,
        inputs=audio_batch,
        device=torch.device("cuda"),
        threshold=0.85  # From threshold optimization
    )

    """
    model.eval()
    inputs = inputs.to(device)
    
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1)
        max_confidence, predicted_class = torch.max(probs, dim=1)

        predictions = []
        confidences = max_confidence.cpu().numpy()

        for conf, pred in zip(confidences, predicted_class.cpu().numpy()):
            if conf >= threshold:
                predictions.append(pred)
            else:
                predictions.append(-1)  # -1 indicates uncertainty

    return predictions, confidences

In [ ]:
model_path = "/kaggle/working/models/autoencoder.pt"
csv_path = "/kaggle/working/csv/ADMR_split"
model_save_path = "/kaggle/working/models/ADMR_model.pt"
generator = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = DeepAutoencoder().to(device)
autoencoder.load_state_dict(torch.load(model_path, map_location=device))

model = ADMR_model(pretrained_autoencoder=autoencoder).to(device)
    
train_set = PathLabelDataset(os.path.join(csv_path, "train.csv"))
val_set = PathLabelDataset(os.path.join(csv_path, "val.csv"))
test_set = PathLabelDataset(os.path.join(csv_path, "test.csv"))

print(f"Train set size: {len(train_set)}\nValidation set size: {len(val_set)}\nTest set size: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)

train_ADMR_model(model, train_loader, val_loader, device, epochs=25, lr=1e-4, save_path=model_save_path)
print(f"ADMR model trained and saved to {model_save_path}")


model.load_state_dict(torch.load(model_save_path, map_location=device))

test_loader = DataLoader(test_set, batch_size=16)
evaluate_model(model, test_loader, device, report_path="/kaggle/working/reports/ADMR/test_report.csv", plot_path="/kaggle/working/plots/ADMR/")
print("Model evaluation completed.")


In [ ]:
compute_confidence_scores_with_preds(model, train_loader, device)
find_confidence_thresholds("/kaggle/working/csv/ADMR/confidence_scores.csv")